# LeetCode #1105 — Filling Bookcase Shelves

**Bài Medium · Python 3 · Quy hoạch động (Dynamic Programming) trên dãy**  
Đề: https://leetcode.com/problems/filling-bookcase-shelves/

```python
class Solution:
    def minHeightShelves(self, books: List[List[int]], shelfWidth: int) -> int:
        ...
```

Notebook dành cho người mới: bắt đầu từ **xếp sách bằng tay**, nhìn thấy vì sao tham lam có thể sai, rồi tự xây dựng **trạng thái và công thức DP**. Mỗi đoạn code đều có ví dụ chạy kiểm tra.

## 1. Mục tiêu học

Bạn sẽ biết:
1. Đọc đúng đề: **giữ nguyên thứ tự sách** và mỗi tầng kệ có tổng độ dày `<= shelfWidth`.
2. Tính chiều cao một tầng kệ bằng chiều cao **cuốn cao nhất** trên tầng đó.
3. Nhận ra vì sao chiến lược "còn chỗ thì nhét tiếp" không luôn tối ưu.
4. Xây dựng `dp[i]` — tổng chiều cao nhỏ nhất để đặt **`i` cuốn đầu tiên**.
5. Viết và kiểm chứng thuật toán `O(n²)`; luyện tập 5 bài biến thể.

## 2. Đề bài bằng tiếng Việt

Mỗi phần tử `books[i] = [thickness_i, height_i]` mô tả **độ dày** và **chiều cao** của cuốn sách thứ `i`.

Các sách phải được đặt **đúng thứ tự ban đầu**. Ta chia dãy thành các đoạn liên tiếp; mỗi đoạn được đặt trên một tầng kệ.

- Tổng độ dày trên mỗi tầng **không vượt quá** `shelfWidth`.
- Chiều cao của một tầng là **chiều cao lớn nhất** trong các sách của tầng.
- Tổng chiều cao tủ là **tổng chiều cao của các tầng**.

**Yêu cầu:** chọn cách ngắt tầng để tổng chiều cao là nhỏ nhất.

**Ví dụ chính thức 1:** `books = [[1,1],[2,3],[2,3],[1,1],[1,1],[1,1],[1,2]]`, `shelfWidth=4` → `6`.

**Ví dụ chính thức 2:** `books = [[1,3],[2,4],[3,2]]`, `shelfWidth=6` → `4`.

**Constraints:** `1 <= len(books) <= 1000`; `1 <= thickness_i <= shelfWidth <= 1000`; `1 <= height_i <= 1000`.

## 3. Bắt đầu bằng ví dụ nhỏ

Xét `books = [[2,1], [1,3], [2,3]]`, `shelfWidth = 3`.

Ta có hai cách:

```text
Cách A: [sách 1, sách 2]  | [sách 3]
          độ dày 2+1=3        độ dày 2
          chiều cao 3        chiều cao 3
          tổng = 3+3 = 6

Cách B: [sách 1]        | [sách 2, sách 3]
          chiều cao 1        chiều cao 3
          tổng = 1+3 = 4  ← tốt hơn
```

Đây là phản ví dụ quan trọng cho cách **tham lam cứ còn chỗ thì đặt tiếp**.

In [1]:
books_demo = [[2, 1], [1, 3], [2, 3]]
width_demo = 3
grouping_a = [books_demo[:2], books_demo[2:]]
grouping_b = [books_demo[:1], books_demo[1:]]

def shelf_height(shelf):
    return max(height for thickness, height in shelf)

def total_height(grouping):
    return sum(shelf_height(shelf) for shelf in grouping)

print('Cách A:', total_height(grouping_a))
print('Cách B:', total_height(grouping_b))
assert total_height(grouping_a) == 6 and total_height(grouping_b) == 4

Cách A: 6
Cách B: 4


## 4. Python nền tảng — list, vòng lặp, `max`, `sum`

Trong `books`, mỗi cuốn là một list gồm `[độ_dày, chiều_cao]`. Với `for thickness, height in books`, Python **tách hai giá trị** và gán lần lượt cho hai biến.

- `sum(...)`: cộng các phần tử.
- `max(...)`: lấy giá trị lớn nhất.
- `books[i:j]`: lấy các cuốn vị trí `i` đến `j-1` (không gồm `j`).

Đây là kiến thức đủ để ta thử mọi cách chia nhỏ, trước khi học DP.

In [2]:
example = [[1, 1], [2, 3], [2, 2]]
for index, (thickness, height) in enumerate(example):
    print(f'Sách {index}: dày={thickness}, cao={height}')
print('Tổng độ dày:', sum(thickness for thickness, _ in example))
print('Chiều cao nếu cùng 1 tầng:', max(h for _, h in example))
print('Hai cuốn đầu:', example[:2])

Sách 0: dày=1, cao=1
Sách 1: dày=2, cao=3
Sách 2: dày=2, cao=2
Tổng độ dày: 5
Chiều cao nếu cùng 1 tầng: 3
Hai cuốn đầu: [[1, 1], [2, 3]]


### 4.1. Tránh nhầm thứ tự hai trường dữ liệu

`books[i][0]` là **thickness**, `books[i][1]` là **height**. Nếu đổi hai trường, bạn có thể tính tổng chiều cao và tổng độ dày sai ngay cả khi code vẫn chạy.

In [3]:
book = [2, 7]
print('Độ dày:', book[0])
print('Chiều cao:', book[1])
assert book[0] == 2 and book[1] == 7

Độ dày: 2
Chiều cao: 7


## 5. Cách trực giác — thử mọi vị trí ngắt tầng

Từ cuốn đầu chưa xếp, chọn **một đoạn liên tiếp** đặt trên tầng đầu. Nếu còn đủ chiều rộng, có thể chọn 1 cuốn, 2 cuốn, 3 cuốn, ...

Sau đó **giải tương tự** cho phần sách còn lại. Ta thử mọi vị trí ngắt và chọn tổng chiều cao nhỏ nhất.

Phương pháp này đúng về mặt ý tưởng nhưng nhiều cấu hình có thể khiến số cách tách dãy tăng **theo hàm mũ** (nếu mọi đoạn đều đặt chung được).

In [4]:
def exhaustive_height(books, shelfWidth):
    """Vét cạn minh hoạ, chỉ dùng n nhỏ."""
    n = len(books)
    def dfs(i):
        if i == n:
            return 0
        width, height = 0, 0
        answer = float('inf')
        for j in range(i, n):
            width += books[j][0]
            if width > shelfWidth:
                break
            height = max(height, books[j][1])
            answer = min(answer, height + dfs(j + 1))
        return answer
    return dfs(0)

assert exhaustive_height(books_demo, width_demo) == 4
print('Vét cạn:', exhaustive_height(books_demo, width_demo))

Vét cạn: 4


### 5.1. Tại sao tham lam không đáng tin?

Một quyết định đặt cuốn hiện tại vào tầng đang có chỗ **có thể làm đổi vị trí ngắt tầng**, từ đó ảnh hưởng tới tất cả cuốn phía sau. Vì thế ta không có cơ sở để kết luận rằng “xếp nhiều nhất có thể ở tầng trước” là tốt nhất.

Hãy tự xem `books_demo`: tham lam dùng **6**, còn phương án tốt nhất dùng **4**.

In [5]:
def greedy_fill_shelves(books, shelfWidth):
    current_width, current_height, total = 0, 0, 0
    for thickness, height in books:
        if current_width + thickness > shelfWidth:
            total += current_height
            current_width = 0
            current_height = 0
        current_width += thickness
        current_height = max(current_height, height)
    return total + current_height

print('Greedy:', greedy_fill_shelves(books_demo, width_demo))
print('Tối ưu:', exhaustive_height(books_demo, width_demo))
assert greedy_fill_shelves(books_demo, width_demo) == 6

Greedy: 6
Tối ưu: 4


## 6. Từ vét cạn tới Dynamic Programming (DP)

**DP** không có nghĩa là thuộc lòng công thức. Ta tìm một *bài toán con* (*subproblem*) xuất hiện lặp lại.

Đặt:

`dp[i] = tổng chiều cao nhỏ nhất để xếp i cuốn đầu tiên`.

Khi đó `dp[0] = 0`: không có sách thì không cần kệ.

Muốn tìm `dp[i]`, hãy đặt câu hỏi **tầng kệ cuối cùng gồm những cuốn nào?**. Nếu tầng cuối chứa các cuốn từ `j` đến `i` (đếm từ 1), thì `j-1` cuốn đầu tiên đã xếp tối ưu với độ cao `dp[j-1]`.

## 7. Công thức chuyển trạng thái — suy từ tầng cuối

Ký hiệu `width(j..i)` là tổng độ dày các cuốn `j..i`, và `height(j..i)` là chiều cao lớn nhất trong đoạn đó.

Với mọi `j` sao cho `width(j..i) <= shelfWidth`:

\[
dp[i] = \min_{1\le j\le i,\,\mathrm{width}(j..i)\le W}
  \Big(dp[j-1] + \mathrm{height}(j..i)\Big).
\]

**Đọc thành lời:** thử mọi cách chọn tầng cuối; tổng = chiều cao tối ưu của đoạn trước + chiều cao tầng cuối; lấy nhỏ nhất.

Cách tính hiệu quả: **giữ điểm cuối `i`, duyệt `j` ngược từ `i` về `1`**; vừa đi vừa cộng độ dày và cập nhật chiều cao lớn nhất. Nếu quá rộng thì `break`.

### 7.1. Vì sao phải đi ngược?

Với `i` cố định, ta muốn mở rộng tầng cuối từ **cuốn i** sang **cuốn i-1**, rồi **cuốn i-2**... Khi lùi `j` một đơn vị, chỉ cần:

`width += books[j-1][0]` và `height = max(height, books[j-1][1])`.

Nếu tính lại `sum` và `max` từ đầu cho mọi đoạn, sẽ tốn công không cần thiết.

In [6]:
# Minh họa: tầng cuối kết thúc ở cuốn số 3 (1-based).
books_small = [[2, 1], [1, 3], [2, 3]]
width, height = 0, 0
i = 3
for j in range(i, 0, -1):
    w, h = books_small[j - 1]
    width += w
    height = max(height, h)
    print(f'Tầng cuối gồm sách {j}..{i}: rộng={width}, cao={height}')
    if width > 3:
        print('→ Quá rộng: không xét được đoạn dài hơn')
        break

Tầng cuối gồm sách 3..3: rộng=2, cao=3
Tầng cuối gồm sách 2..3: rộng=3, cao=3
Tầng cuối gồm sách 1..3: rộng=5, cao=3
→ Quá rộng: không xét được đoạn dài hơn


## 8. Cú pháp Python cho DP: `float('inf')`, `range`, `break`

- `float('inf')` biểu diễn một giá trị lớn hơn mọi số hữu hạn; dùng khi tìm **minimum**.
- `range(i, 0, -1)` tạo `i, i-1, ..., 1`.
- `break` kết thúc **vòng for gần nhất** khi tổng độ dày vượt giới hạn.

Ta chỉ cần một list `dp` dài `n+1`. Không cần tạo ma trận 2 chiều.

In [7]:
print('Vô cực > 1000?', float('inf') > 1000)
print('Duyệt ngược:', list(range(5, 0, -1)))
dp_demo = [float('inf')] * 4
dp_demo[0] = 0
print('Mảng DP khởi tạo:', dp_demo)

Vô cực > 1000? True
Duyệt ngược: [5, 4, 3, 2, 1]
Mảng DP khởi tạo: [0, inf, inf, inf]


## 9. Cách tối ưu — Bottom-up DP

**Bottom-up**: lần lượt tính từ `dp[0]` đến `dp[n]`.

Với từng số sách `i`, thử chọn tầng cuối bắt đầu ở `j=i, i-1, ..., 1`. Mỗi lần mở rộng đoạn, tính chiều cao:

`candidate = dp[j - 1] + max_height`.

Khi độ dày vượt giới hạn, dừng duyệt `j` vì thêm sách về bên trái chỉ làm kệ dày hơn.

In [8]:
def min_height_shelves_dp(books, shelfWidth):
    n = len(books)
    dp = [0] + [float('inf')] * n
    for i in range(1, n + 1):
        width = 0
        tallest = 0
        for j in range(i, 0, -1):
            thickness, height = books[j - 1]
            width += thickness
            if width > shelfWidth:
                break
            tallest = max(tallest, height)
            dp[i] = min(dp[i], dp[j - 1] + tallest)
    return dp[n]

print(min_height_shelves_dp(books_demo, width_demo))
assert min_height_shelves_dp(books_demo, width_demo) == 4

4


## 10. TRACE — nhìn giá trị `dp` thay đổi

Với `books=[[2,1],[1,3],[2,3]]`, `shelfWidth=3`:

| `i` (số cuốn đã xét) | Cách chọn tầng cuối khả thi | Giá trị nhỏ nhất `dp[i]` |
|---:|---|---:|
| 0 | Không có sách | 0 |
| 1 | `[1]` có chiều cao 1 | 1 |
| 2 | `[2]` + `dp[1]` = 4; `[1,2]` cao 3 | 3 |
| 3 | `[3]` + `dp[2]` = 6; `[2,3]` cao 3 + `dp[1]` = 4 | **4** |

Quan trọng: `dp[2]=3` không nhất thiết đồng nghĩa với việc chia tối ưu ở `i=3` phải tiếp tục từ cách đã xếp 2 cuốn. DP **thử lại vị trí bắt đầu tầng cuối**.

In [9]:
def trace_shelves_dp(books, shelfWidth):
    dp = [0] + [float('inf')] * len(books)
    for i in range(1, len(books) + 1):
        width = height = 0
        print(f'\ni={i}: tìm cách xếp {i} sách đầu')
        for j in range(i, 0, -1):
            width += books[j - 1][0]
            if width > shelfWidth:
                print(f'  j={j}: rộng={width} > {shelfWidth}, dừng')
                break
            height = max(height, books[j - 1][1])
            candidate = dp[j - 1] + height
            dp[i] = min(dp[i], candidate)
            print(f'  j={j}: rộng={width}, cao={height}, '
                  f'dp[{j-1}]+cao={candidate}, dp[{i}]={dp[i]}')
    print('\nToàn bộ dp:', dp)
    return dp[-1]

assert trace_shelves_dp(books_demo, 3) == 4


i=1: tìm cách xếp 1 sách đầu
  j=1: rộng=2, cao=1, dp[0]+cao=1, dp[1]=1

i=2: tìm cách xếp 2 sách đầu
  j=2: rộng=1, cao=3, dp[1]+cao=4, dp[2]=4
  j=1: rộng=3, cao=3, dp[0]+cao=3, dp[2]=3

i=3: tìm cách xếp 3 sách đầu
  j=3: rộng=2, cao=3, dp[2]+cao=6, dp[3]=6
  j=2: rộng=3, cao=3, dp[1]+cao=4, dp[3]=4
  j=1: rộng=5 > 3, dừng

Toàn bộ dp: [0, 1, 3, 4]


## 11. Phiên bản LeetCode — chỉ cần hàm ngắn gọn

Không `print` và không mang code trace vào bài nộp. Dùng đúng chữ ký `minHeightShelves(self, books, shelfWidth)`.

In [10]:
from typing import List

class Solution:
    def minHeightShelves(self, books: List[List[int]], shelfWidth: int) -> int:
        n = len(books)
        dp = [0] + [float('inf')] * n
        for i in range(1, n + 1):
            width = 0
            height = 0
            for j in range(i, 0, -1):
                width += books[j - 1][0]
                if width > shelfWidth:
                    break
                height = max(height, books[j - 1][1])
                dp[i] = min(dp[i], dp[j - 1] + height)
        return dp[n]

ex1 = [[1,1],[2,3],[2,3],[1,1],[1,1],[1,1],[1,2]]
ex2 = [[1,3],[2,4],[3,2]]
assert Solution().minHeightShelves(ex1, 4) == 6
assert Solution().minHeightShelves(ex2, 6) == 4
print('Đã PASS 2 ví dụ chính thức')

Đã PASS 2 ví dụ chính thức


## 12. Tại sao công thức DP đúng?

Chứng minh bằng **quy nạp theo `i`**.

**Cơ sở:** `i=0`, không có sách nên chiều cao tối ưu bằng `0`.

**Giả sử:** `dp[0]..dp[i-1]` đều tối ưu. Với một cách xếp hợp lệ bất kỳ của `i` cuốn, tầng cuối phải là **một đoạn liên tiếp `j..i`** vì không được đảo thứ tự. Phần `1..j-1` phải có tổng chiều cao ít nhất `dp[j-1]`, theo giả thiết quy nạp.

Vậy bất kỳ phương án nào cũng có độ cao ít nhất `dp[j-1]+height(j..i)` với một `j` khả thi. Thuật toán thử **tất cả `j` khả thi** và chọn giá trị nhỏ nhất; mỗi lựa chọn lại ghép được thành một phương án hợp lệ. Do đó `dp[i]` vừa không nhỏ hơn vừa không lớn hơn tối ưu → **chính là tối ưu**.

## 13. Phân tích độ phức tạp

Gọi `n = len(books)`, `W = shelfWidth`.

- Với mỗi `i` (tối đa `n` giá trị), vòng `j` lùi qua **tối đa `i` cuốn**. Mỗi lần cập nhật tốn `O(1)` → thời gian `O(n²)` trong trường hợp xấu nhất.
- `dp` dài `n+1` → bộ nhớ `O(n)`.
- Baseline vét cạn có thể có `2^(n-1)` kiểu ngắt giữa `n` cuốn (nếu mọi đoạn đều vừa kệ), nên không dùng để submit.

`n <= 1000`, DP `O(n²)` chạy phù hợp; không cần DP 2 chiều.

## 14. Cách thứ hai — Top-down với `lru_cache` (để hiểu memoization)

Thay vì tính `dp[i]` từ trái qua phải, gọi `f(i)` là chiều cao nhỏ nhất để xếp sách từ chỉ số `i` **đến cuối**.

Chọn `j` là cuốn cuối tầng đầu, rồi cộng `f(j+1)`. `@lru_cache(None)` ghi nhớ kết quả của từng `i` để không tính lặp. Thời gian `O(n²)`, bộ nhớ `O(n)` cho cache và stack.

**Lưu ý:** Python có giới hạn độ sâu đệ quy; `n` có thể bằng 1000 nên **chọn bottom-up làm bản submit** an toàn hơn. Code dưới dùng cho các ví dụ vừa/nhỏ.

In [11]:
from functools import lru_cache

def min_height_shelves_top_down(books, shelfWidth):
    n = len(books)
    @lru_cache(None)
    def f(i):
        if i == n:
            return 0
        width = height = 0
        best = float('inf')
        for j in range(i, n):
            width += books[j][0]
            if width > shelfWidth:
                break
            height = max(height, books[j][1])
            best = min(best, height + f(j + 1))
        return best
    return f(0)

assert min_height_shelves_top_down(books_demo, 3) == 4
assert min_height_shelves_top_down(ex1, 4) == 6
print('PASS: Top-down = Bottom-up')

PASS: Top-down = Bottom-up


## 15. Biến thể hữu ích — khôi phục cách chia tầng tối ưu

Bài gốc chỉ cần **chiều cao nhỏ nhất**, nhưng đôi khi bạn muốn biết **ngắt tầng ở đâu**.

Trong lúc cập nhật `dp[i]`, lưu thêm `prev[i] = j-1`: điểm kết thúc của phần sách trước tầng cuối. Sau khi tính xong, đi từ `i=n` ngược về `0` để dựng lại các đoạn sách.

Đây là kỹ thuật rất thường gặp: **DP tính giá trị tối ưu + truy vết lựa chọn** (*backtracking the decisions*).

In [12]:
def shelves_with_partition(books, shelfWidth):
    n = len(books)
    dp = [0] + [float('inf')] * n
    prev = [-1] * (n + 1)
    for i in range(1, n + 1):
        width = height = 0
        for j in range(i, 0, -1):
            width += books[j - 1][0]
            if width > shelfWidth:
                break
            height = max(height, books[j - 1][1])
            candidate = dp[j - 1] + height
            if candidate < dp[i]:
                dp[i] = candidate
                prev[i] = j - 1
    partitions = []
    i = n
    while i > 0:
        j = prev[i]
        partitions.append(books[j:i])
        i = j
    partitions.reverse()
    return dp[n], partitions

score, layout = shelves_with_partition(books_demo, 3)
print('Chiều cao:', score, '| Các tầng:', layout)
assert score == 4 and total_height(layout) == score

Chiều cao: 4 | Các tầng: [[[2, 1]], [[1, 3], [2, 3]]]


## 16. Bộ test quan trọng

- Một cuốn duy nhất.
- Tất cả cùng một tầng (đủ chỗ).
- Mỗi cuốn bắt buộc một tầng (mỗi sách có độ dày bằng `shelfWidth`).
- Sách rất cao giữa các cuốn thấp, dễ làm tham lam sai.
- Bộ test ngẫu nhiên nhỏ: so sánh với **vét cạn độc lập**.

In [13]:
samples = [
    ([[1, 5]], 1, 5),
    ([[1, 2], [1, 3], [1, 1]], 3, 3),
    ([[2, 3], [2, 6], [2, 1]], 2, 10),
    ([[2, 1], [1, 3], [2, 3]], 3, 4),
    (ex1, 4, 6),
    (ex2, 6, 4),
]
for books, width, expected in samples:
    actual = Solution().minHeightShelves(books, width)
    assert actual == expected, (books, actual, expected)
print('PASS:', len(samples), 'ca cố định')

PASS: 6 ca cố định


In [14]:
from random import Random
rng = Random(1105)
count = 0
for _ in range(250):
    width = rng.randint(1, 8)
    books = [[rng.randint(1, width), rng.randint(1, 9)]
             for _ in range(rng.randint(1, 8))]
    expected = exhaustive_height(books, width)
    actual = Solution().minHeightShelves(books, width)
    alternative = min_height_shelves_top_down(books, width)
    assert actual == expected == alternative, (books, width, actual, expected)
    count += 1
print('PASS:', count, 'ca ngẫu nhiên đối chiếu 3 cách')

PASS: 250 ca ngẫu nhiên đối chiếu 3 cách


In [15]:
# Kiểm thử ngay ở biên chính thức n = 1000.
big_books = [[1, (i * 17) % 1000 + 1] for i in range(1000)]
assert Solution().minHeightShelves(big_books, 1000) == max(h for _, h in big_books)
big_books2 = [[1000, 1 + i % 1000] for i in range(1000)]
assert Solution().minHeightShelves(big_books2, 1000) == sum(h for _, h in big_books2)
print('PASS: 2 ca n=1000')

PASS: 2 ca n=1000


## 17. Sai lầm hay gặp

**Sai 1:** cho phép **đảo thứ tự sách** để tìm đáp án nhỏ hơn. Đề buộc phải giữ thứ tự; các tầng là **đoạn liên tiếp**.

**Sai 2:** cộng chiều cao **mọi** cuốn trên cùng tầng. Một tầng chỉ tăng chiều cao bằng `max(height)`.

**Sai 3:** quên `dp[0] = 0`. Khi tầng cuối lấy từ cuốn 1, công thức cần `dp[0]`.

**Sai 4:** chỉ xét các đoạn có độ dày `< shelfWidth`. Điều kiện đúng là `<= shelfWidth`.

**Sai 5:** khi mở rộng tầng cuối về bên trái, chỉ cập nhật `width` mà quên `height = max(height, ...)`.

In [16]:
wrong_sum_of_heights = 1 + 3
right_max_height = max(1, 3)
print('Nếu 2 cuốn chung tầng: sai (sum)=', wrong_sum_of_heights,
      ', đúng (max)=', right_max_height)
assert right_max_height == 3
assert Solution().minHeightShelves([[2, 1], [1, 3]], 3) == 3  # Vừa đúng width.

Nếu 2 cuốn chung tầng: sai (sum)= 4 , đúng (max)= 3


### 17.1. Sai `range` khiến bỏ lỡ lựa chọn đầu tầng

Vòng lặp `for j in range(i, 0, -1)` **phải xét cả `j=1`**. Nếu dùng `range(i, 1, -1)`, ta bỏ lỡ phương án đặt tất cả `i` cuốn lên cùng một tầng.

In [17]:
print('Đúng:', list(range(3, 0, -1)))
print('Sai :', list(range(3, 1, -1)))
assert 1 in range(3, 0, -1) and 1 not in range(3, 1, -1)

Đúng: [3, 2, 1]
Sai : [3, 2]


## 18. Nhận diện DP kiểu "chia dãy thành các đoạn liên tiếp"

Dấu hiệu thường thấy:
- Phải giữ nguyên thứ tự phần tử.
- Chọn **vị trí ngắt** để gom các phần tử thành từng nhóm/đoạn.
- Mỗi nhóm có **chi phí riêng** (ở đây là chiều cao lớn nhất).
- Cần tổng chi phí nhỏ nhất hoặc lớn nhất.

Các bài tương tự là phân đoạn mảng, phân bổ công việc theo ngày hoặc chia chuỗi thành đoạn. Mẫu tư duy: **xét nhóm cuối + lời giải tối ưu của phần trước**.

## 19. Bài tập tự luyện (5 mức)

**Bài 1 — Tính tay:** với `books=[[1,2],[2,5]]`, `W=3`, tính chiều cao nếu xếp chung và nếu tách tầng; chọn nhỏ nhất.

**Bài 2 — Đoán output:** `list(range(4,0,-1))`; `max(3,1,7)`; `float('inf') > 10**100`.

**Bài 3 — TODO:** viết lại `exercise_min_height` bằng DP, không được gọi lời giải sẵn.

**Bài 4 — TODO:** viết `exercise_shelf_width(books)` trả **độ dày lớn nhất** của một cuốn, tức chiều rộng kệ nhỏ nhất để có nghiệm hợp lệ.

**Bài 5 — Biến thể:** viết `exercise_reconstruct` trả `(chiều cao tối ưu, danh sách đoạn sách theo tầng)`; cho phép nhiều phương án tối ưu, chỉ kiểm đúng tổng chiều cao và độ rộng mỗi tầng.

In [18]:
def exercise_min_height(books, shelfWidth):
    # TODO: đặt dp[0]=0; duyệt i tăng dần; duyệt j ngược để chọn tầng cuối.
    pass

def exercise_shelf_width(books):
    # TODO: dùng max trên thickness.
    pass

def exercise_reconstruct(books, shelfWidth):
    # TODO: lưu prev[i] khi cập nhật dp[i], rồi đi ngược từ n về 0.
    pass

# Tự làm xong rồi mới bỏ comment:
# assert exercise_min_height(books_demo, 3) == 4
# assert exercise_shelf_width([[2,1],[4,3]]) == 4
# best, layout = exercise_reconstruct(books_demo, 3)
# assert best == 4

## 20. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Tầng chung cao bằng `max`, tách tầng thì cộng hai chiều cao | `max(2,5)` | Đáp án `5` |
| 2 | `range` có bước `-1` | Bắt đầu từ `4`, dừng trước `0` | `[4,3,2,1]` |
| 3 | Chọn **đoạn cuối**, không chọn cuốn kế tiếp theo tham lam | `dp=[0]+[inf]*n` | `dp[i]=min(dp[i],dp[j-1]+height)` |
| 4 | Kệ phải chứa được cuốn dày nhất | `max(...)` | `max(w for w,h in books)` |
| 5 | Nhớ vị trí bắt đầu đoạn cuối | `prev[i]=j-1` | Lần theo `prev` từ `n` về `0` |

## 21. Đáp án tham khảo (đặt sau bài tập)

Bài 1: Xếp chung dày `1+2=3` không vượt giới hạn; cao `max(2,5)=5`. Tách ra cao `2+5=7` → **5**.

Bài 2: `[4,3,2,1]`, `7`, `True`.

In [19]:
def answer_min_height(books, shelfWidth):
    dp = [0] + [float('inf')] * len(books)
    for i in range(1, len(books) + 1):
        used = tallest = 0
        for j in range(i, 0, -1):
            used += books[j - 1][0]
            if used > shelfWidth:
                break
            tallest = max(tallest, books[j - 1][1])
            dp[i] = min(dp[i], dp[j - 1] + tallest)
    return dp[-1]

def answer_shelf_width(books):
    return max(w for w, h in books)

def answer_reconstruct(books, shelfWidth):
    return shelves_with_partition(books, shelfWidth)

assert answer_min_height(books_demo, 3) == 4
assert answer_shelf_width([[2, 1], [4, 3]]) == 4
best, grouping = answer_reconstruct(books_demo, 3)
assert best == total_height(grouping) == 4
assert all(sum(w for w, _ in shelf) <= 3 for shelf in grouping)
assert [b for shelf in grouping for b in shelf] == books_demo
print('PASS: lời giải 3 bài tập code')

PASS: lời giải 3 bài tập code


## 22. Template nhớ nhanh và checklist nộp bài

```python
dp[0] = 0
for i in range(1, n+1):
    width = height = 0
    for j in range(i, 0, -1):
        width += books[j-1][0]
        if width > shelfWidth: break
        height = max(height, books[j-1][1])
        dp[i] = min(dp[i], dp[j-1] + height)
```

- [ ] Tôi biết `dp[i]` biểu thị **i cuốn đầu**, không phải cuốn thứ i.
- [ ] Tôi giữ thứ tự sách và chỉ xét các đoạn liên tiếp.
- [ ] Tôi lấy **max chiều cao** trên một tầng.
- [ ] Tôi hiểu vì sao phải duyệt lại tất cả điểm bắt đầu `j` hợp lệ.
- [ ] Tôi kiểm tra `n=1`, mọi cuốn cùng tầng, mỗi cuốn một tầng, phản ví dụ greedy.
- [ ] Complexity đúng: `O(n²)` thời gian và `O(n)` bộ nhớ.

**Câu chốt:** muốn tối ưu cả dãy, thử **tầng cuối là đoạn nào** và kết hợp lời giải tối ưu cho phần trước.